# Silver to gold

In [1]:
import pandas as pd
from datetime import datetime
import logging
from notebookutils import mssparkutils
import fsspec


In [ ]:
# ===============================
# directory parameters
# ===============================

CONTAINER_NAME = "bee-haven-data-lake-container"
STORAGE_ACCOUNT = "beehavenhivedatalake"


root_folder = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
gold_sink = root_folder+'3. gold/'


In [2]:
# ===============================
# Logger
# ===============================



logging.getLogger('azure.core.pipeline.policies.http_logging_policy').setLevel(logging.WARNING)
logging.getLogger('azure.storage').setLevel(logging.WARNING)
logging.getLogger('msal').setLevel(logging.WARNING)



logger= logging.getLogger(__name__)

logfile_str = f'{gold_sink}/Silver_to_Gold_log_{datetime.now().strftime("%Y%m%d-%H%M%S")}.log'

memory_handler = logging.handlers.MemoryHandler(capacity=10000, target=None)
memory_handler.setFormatter(logging.Formatter('%(asctime)s-%(levelname)s-%(message)s'))

root_logger = logging.getLogger()
root_logger.setLevel(logging.INFO)
root_logger.addHandler(memory_handler)




In [ ]:
measure_list = ['flow',
                'temperature',
                'weight',
                'humidity',
                'weather'
                ]

In [ ]:
# ===============================
# Get file info
# ===============================



def get_files(measure_list:list, root_folder:str) -> dict:

    files = {}

    for measure in measure_list:

        try:

            folder = root_folder+"silver/"+measure+"/"
            file_list = mssparkutils.fs.ls(folder)

            for file in file_list:

                files.setdefault(measure, []).append(file)


        except Exception:

            logger.exception(f"{measure} not found in file directory: {root_folder}/silver/")
            raise

    logger.info(f"files gathered: {files}")

    return files

# ===============================
# load dataframes
# ===============================



def get_df(files) -> dict:

    df_dict = {}

    for measure, file_list in files.items():

        for file in file_list:
            if file.name.endswith(".parquet"):

                try:
                    location = file.name.split("_")[0]
                    df_dict.setdefault(location, {})
                    df_dict[location].setdefault(measure, [])
                    measure_df = pd.read_parquet(file.path)
                    df_dict[location][measure].append(measure_df)


                    logger.info(f'loaded {file.name} dataframe successfully')

                except Exception as e:
                    logger.exception(f"Failed processing file: {file.path}. Error: {str(e)}")

    return df_dict


# ===============================
# in case of multiple dataframes of same measure, concat them
# ===============================


def concat_measures(df_dict:dict) -> dict:

    logger.info(f"starting concatenating dataframes")


    for location, measures in df_dict.items():
        for measure, df_list in measures.items():
            logger.info(f'concatenating {len(df_dict[location][measure])} into one dataframe for {location} - {measure}')
            concat_df = pd.concat(df_list, axis=0, ignore_index=True)
            concat_df.sort_values(by='timestamp', inplace=True)
            df_dict[location][measure] = concat_df

    return df_dict


# ===============================
# Merging dataframes
# ===============================


def merge_measures(df_dict:dict) -> dict:

    logger.info(f"starting merging dataframes")


    df_map = {
    "timestamp": "datetime64[ns, UTC]",
    "flow_out": "float64",
    "flow_in": "float64",
    "temperature_hive": "float64",
    "weight": "float64",
    "humidity_hive": "float64",
    'source_id': 'Int64',
    'temperature_station': 'float64',
    'precipitation': 'float64',
    'pressure_msl': 'float64',
    'sunshine': 'float64',
    'wind_direction': 'float64',
    'wind_speed': 'float64',
    'wind_gust_direction': 'float64',
    'wind_gust_speed': 'float64',
    'cloud_cover': 'float64',
    'dew_point': 'float64',
    'relative_humidity': 'float64',
    'visibility': 'float64',
    'solar': 'float64',
    'precipitation_probability': 'float64',
    'condition': 'object',
    'icon': 'object'}


    for location, measures in df_dict.items():

        df_list = list(measures.values())
        merged_df = df_list[0]

        for df in df_list[1:]:
            merged_df = merged_df.merge(df, on='timestamp', how='outer')

        merged_df = merged_df.reindex(columns=list(df_map.keys())).astype(df_map)


        df_dict[location] = merged_df

        logger.info(f'merged {location} dataframe successfully')

    return df_dict


# ===============================
# define writing to gold
# ===============================


def write_to_gold(df_dict:dict):

    logger.info(f"starting writing to gold...")

    for location, df in df_dict.items():

        filestr = gold_sink+location+f"_measures_merged_{datetime.now().strftime('%Y-%m-%d_T%H-%M-%S')}.parquet"

        with fsspec.open(filestr, 'wb') as folder:
            df.to_parquet(folder, index=False)

            logger.info(f'wrote gold file {filestr} successfully')


In [ ]:
# ===============================
# define main
# ===============================


def main(measure_list:list, root_folder:str) -> None:
    files = get_files(measure_list, root_folder)
    df_dict = get_df(files)
    df_dict = concat_measures(df_dict)
    df_dict = merge_measures(df_dict)
    write_to_gold(df_dict)

    logger.info(f"finished writing to gold, pipeline run success!")


In [ ]:
# ===============================
# Main Execution
# ===============================
main(measure_list, root_folder)

log_stream = fsspec.open(logfile_str, 'w').open()

try:
    memory_handler.target = logging.StreamHandler(log_stream)
    memory_handler.flush()
finally:
    log_stream.close()